# HTD OOF v2 Qwen3-VL 8B OCR LoRA

Rented-GPU notebook. Put `kaggle.json` and the three fold jsonl files next to this notebook, then run all cells. The notebook downloads the Kaggle dataset and Qwen3-VL model to `./data/rukopys-dataset` and `./models/Qwen3-VL-8B-Instruct`, crops bbox regions in memory during training, and saves the final LoRA under `./outputs/htd_oof_v2_qwen3vl_ocr/...`.

In [ ]:
# Rented GPU dependency cell.
INSTALL_DEPS = True

if INSTALL_DEPS:
    import subprocess
    import sys

    commands = [
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade-strategy",
            "only-if-needed",
            "accelerate",
            "peft",
            "bitsandbytes",
            "trl",
            "qwen-vl-utils",
            "datasets",
            "hf_transfer",
            "huggingface_hub",
            "kaggle",
            "pandas==2.2.2",
            "pillow<12",
        ],
        [sys.executable, "-m", "pip", "install", "-q", "-U", "git+https://github.com/huggingface/transformers.git"],
    ]
    for cmd in commands:
        print("Running:", " ".join(cmd), flush=True)
        subprocess.check_call(cmd)

try:
    import subprocess
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("nvidia-smi not found. Make sure this rented machine has NVIDIA drivers and a CUDA GPU.", flush=True)


In [ ]:
import gc
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import torch
from datasets import Dataset
from PIL import Image

Image.MAX_IMAGE_PIXELS = None
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
warnings.filterwarnings("ignore", message=r".*Kwargs passed to `processor\.__call__` have to be in `processor_kwargs` dict.*")

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required. Enable the rented GPU environment before running this notebook.")
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

# =========================
# EDIT ONLY THIS BLOCK
# =========================
# Run the notebook from the folder that contains:
#   kaggle.json
#   ScannedV1-FOLD1.jsonl
#   ScannedV1-FOLD2.jsonl
#   ScannedV1-FOLD3.jsonl
# If your Jupyter process starts elsewhere, set HTD_NOTEBOOK_DIR to that folder before running.
NOTEBOOK_DIR = Path(os.environ.get("HTD_NOTEBOOK_DIR", ".")).expanduser().resolve()

# MODEL_ID follows the old v1 OOF mapping:
#   1 trains FOLD1 + FOLD2, holds out FOLD3.
#   2 trains FOLD1 + FOLD3, holds out FOLD2.
#   3 trains FOLD2 + FOLD3, holds out FOLD1.
MODEL_ID = int(os.environ.get("HTD_MODEL_ID", "1"))

# Default slug used by the older notebooks in this repo. Change it if your Kaggle dataset slug is different.
KAGGLE_DATASET_SLUG = os.environ.get("KAGGLE_DATASET_SLUG", "quii29/rukopys-dataset").strip()
KAGGLE_JSON_PATH = Path(os.environ.get("KAGGLE_JSON_PATH", str(NOTEBOOK_DIR / "kaggle.json"))).expanduser().resolve()
DATASET_DOWNLOAD_DIR = Path(os.environ.get("HTD_DATASET_DOWNLOAD_DIR", str(NOTEBOOK_DIR / "data" / "rukopys-dataset"))).expanduser().resolve()
FORCE_DATASET_DOWNLOAD = bool(int(os.environ.get("HTD_FORCE_DATASET_DOWNLOAD", "0")))

# Leave empty to use DATASET_DOWNLOAD_DIR. Set only if the dataset is already unpacked elsewhere.
DATASET_ROOT_OVERRIDE = os.environ.get("HTD_DATASET_ROOT", "").strip()
FOLD_JSONL_ROOT = Path(os.environ.get("HTD_FOLD_JSONL_ROOT", str(NOTEBOOK_DIR))).expanduser().resolve()

QWEN_MODEL_ID = os.environ.get("HTD_QWEN_MODEL_ID", "Qwen/Qwen3-VL-8B-Instruct").strip()
MODEL_DOWNLOAD_DIR = Path(
    os.environ.get("HTD_MODEL_DOWNLOAD_DIR", str(NOTEBOOK_DIR / "models" / "Qwen3-VL-8B-Instruct"))
).expanduser().resolve()
FORCE_MODEL_DOWNLOAD = bool(int(os.environ.get("HTD_FORCE_MODEL_DOWNLOAD", "0")))
START_LORA_DIR = os.environ.get("HTD_START_LORA_DIR", "").strip()

NUM_TRAIN_EPOCHS = 3
VALIDATION_FRACTION = 0.15
TEST_MODE = False
TEST_MAX_SAMPLES = 256

# A6000 speed settings. MAX_SPEED_MODE probes the largest batch that fits and uses grad_accum=1.
MAX_SPEED_MODE = bool(int(os.environ.get("HTD_MAX_SPEED_MODE", "1")))
AUTO_TUNE_BATCH_SIZE = bool(int(os.environ.get("HTD_AUTO_TUNE_BATCH_SIZE", "1")))
BATCH_SIZE_CANDIDATES = [int(x) for x in os.environ.get("HTD_BATCH_SIZE_CANDIDATES", "16,12,10,8,6,4,2,1").split(",") if x.strip()]
TARGET_EFFECTIVE_BATCH = int(os.environ.get("HTD_TARGET_EFFECTIVE_BATCH", "128"))
PER_DEVICE_BATCH = int(os.environ.get("HTD_PER_DEVICE_BATCH", "8"))
GRAD_ACCUM = int(os.environ.get("HTD_GRAD_ACCUM", "1" if MAX_SPEED_MODE else "32"))
DATALOADER_NUM_WORKERS = int(os.environ.get("HTD_DATALOADER_WORKERS", "2"))
USE_GRADIENT_CHECKPOINTING = bool(int(os.environ.get("HTD_GRADIENT_CHECKPOINTING", "0")))
MAX_SEQ_LENGTH = 3072
MAX_PIXELS_CROP = 320_000
CROP_PAD_RATIO = 0.04

LEARNING_RATE = 1e-4
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

LOGGING_STEPS = 1
SAVE_STEPS = 80
SAVE_TOTAL_LIMIT = 2
RESUME_TRAINING = bool(int(os.environ.get("HTD_RESUME_TRAINING", "0")))
RESUME_CHECKPOINT_DIR = os.environ.get("HTD_RESUME_CHECKPOINT_DIR", "").strip()

OUTPUT_ROOT = Path(os.environ.get("HTD_OUTPUT_ROOT", str(NOTEBOOK_DIR / "outputs" / "htd_oof_v2_qwen3vl_ocr"))).expanduser().resolve()
# =========================
# END EDIT BLOCK
# =========================

MODEL_TO_TRAIN_FOLDS = {1: [1, 2], 2: [1, 3], 3: [2, 3]}
FOLD_FILE_CANDIDATES = {
    1: ["ScannedV1-FOLD1.jsonl", "metadata_part1.jsonl"],
    2: ["ScannedV1-FOLD2.jsonl", "metadata_part2.jsonl"],
    3: ["ScannedV1-FOLD3.jsonl", "metadata_part3.jsonl"],
}
if MODEL_ID not in MODEL_TO_TRAIN_FOLDS:
    raise ValueError(f"MODEL_ID must be one of {sorted(MODEL_TO_TRAIN_FOLDS)}, got {MODEL_ID}")
TRAIN_FOLDS = MODEL_TO_TRAIN_FOLDS[MODEL_ID]
HELD_OUT_FOLD = sorted(set(FOLD_FILE_CANDIDATES) - set(TRAIN_FOLDS))[0]
RUN_NAME = f"model{MODEL_ID}_train_folds{'_'.join(map(str, TRAIN_FOLDS))}_heldout_fold{HELD_OUT_FOLD}"

MODEL_OUTPUT_DIR = OUTPUT_ROOT / RUN_NAME
TRAINER_OUTPUT_DIR = MODEL_OUTPUT_DIR / "trainer_checkpoints"
FINAL_DIR = MODEL_OUTPUT_DIR / f"qwen3vl_oof_v2_ocr_{RUN_NAME}_lora_final"
for path in [OUTPUT_ROOT, MODEL_OUTPUT_DIR, TRAINER_OUTPUT_DIR, FINAL_DIR, DATASET_DOWNLOAD_DIR, MODEL_DOWNLOAD_DIR.parent]:
    path.mkdir(parents=True, exist_ok=True)

GPU_TOTAL_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3
CUDA_MAJOR = torch.cuda.get_device_capability(0)[0]
USE_BF16 = CUDA_MAJOR >= 8
TORCH_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
MODEL_DEVICE_MAP = "auto"

OCR_TYPES = {"handwritten", "printed", "formula", "table", "annotation"}
VALID_TYPES = {"handwritten", "printed", "formula", "table", "annotation", "image", "graph"}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}

PROMPT_VERSION = "oof_v2_hybrid_prompt_v3"

SPECIAL_TEXT_MARKER_RULES = (
    "Use these special markers only when they are visible in the crop: "
    "~~word~~ for strikethrough text, ~~old~~{new} for strikethrough text with a visible correction, "
    "and [illegible] for an unreadable word inside an otherwise legible line. "
)

DOCUMENT_CONTEXT_RULES = (
    "The crop may come from Ukrainian dictation handwriting, historical Ukrainian/Cyrillic archives, "
    "manuscripts, school homework, exams, university coursework, lecture notes, scientific documents, "
    "tables, formulas, chemistry notation, teacher marks, maps, diagrams, or mixed handwriting and print. "
    "Read only what is visually present in the image. Do not complete from canonical text, memorized poems, "
    "known exercises, surrounding context, or likely answers. Preserve old spelling, original orthography, "
    "visible spelling mistakes, capitalization, punctuation, line breaks, corrections, and teacher marks. "
)

COMMON_OCR_RULES = (
    "Return only the transcription. No JSON, no Markdown, no explanation. "
    + DOCUMENT_CONTEXT_RULES
    + SPECIAL_TEXT_MARKER_RULES
    + "Keep uncertain characters exactly as seen. Do not translate, correct grammar, normalize spelling, "
    "expand abbreviations, summarize, infer hidden/missing text, or silently replace unreadable words. "
)

CROP_PROMPTS = {
    "handwritten": (
        "Transcribe the visible handwritten text exactly. Preserve line content, punctuation, corrections, "
        "crossed-out text, insertions, abbreviations, digits, units, and original Ukrainian/Cyrillic spelling. "
        + COMMON_OCR_RULES
    ),
    "printed": (
        "Transcribe the visible printed or typed text exactly. Preserve line content, punctuation, numbering, "
        "abbreviations, capitalization, hyphenation, old spelling, and visible corrections. "
        + COMMON_OCR_RULES
    ),
    "annotation": (
        "Read this short annotation, teacher mark, grade, correction, marginal note, numbering, or stamp. "
        "Return only the exact visible text or mark. "
        + COMMON_OCR_RULES
    ),
    "formula": (
        "Read this standalone math, logic, vector, matrix, determinant, set/relation, statistics, physics, "
        "or chemistry expression exactly as written. Return only formula text, using LaTeX when it is the "
        "clearest representation and plain Unicode when it better matches the handwriting. Do not wrap the "
        "answer in dollar signs. Preserve visible symbols, indices, superscripts, subscripts, arrows, fractions, "
        "matrix/determinant structure, punctuation, numbering, units, strikethroughs, and correction markers. "
        + SPECIAL_TEXT_MARKER_RULES
        + "Do not solve, simplify, normalize, explain, or convert old notation into a different style. If unreadable, return [illegible]."
    ),
    "table": (
        "Read this table region exactly. Return only pipe-separated table text. Use one output line per visual row "
        "and | between cells. Preserve empty cells with empty fields, for example A||C. Preserve row order, "
        "column order, wrapped cell text, numbers, units, punctuation, dashes, spelling mistakes, corrections, "
        "and strikethrough markers. "
        + SPECIAL_TEXT_MARKER_RULES
        + "Do not infer missing cells, rebalance columns, summarize, or explain. If unreadable, return [illegible]."
    ),
    "image": "Return an empty string.",
    "graph": "Return an empty string.",
    "default": "Transcribe the visible content exactly. " + COMMON_OCR_RULES,
}

print("Config OK")
print("GPU:", torch.cuda.get_device_name(0), f"{GPU_TOTAL_GIB:.1f} GiB")
print("Notebook dir:", NOTEBOOK_DIR)
print("Kaggle JSON:", KAGGLE_JSON_PATH)
print("Dataset download dir:", DATASET_DOWNLOAD_DIR)
print("Qwen model id:", QWEN_MODEL_ID)
print("Qwen model local dir:", MODEL_DOWNLOAD_DIR)
print("MODEL_ID:", MODEL_ID, "train folds:", TRAIN_FOLDS, "held out:", HELD_OUT_FOLD)
print("Run name:", RUN_NAME)
print("Torch dtype:", TORCH_DTYPE)
print("Max speed mode:", MAX_SPEED_MODE)
print("Auto tune batch size:", AUTO_TUNE_BATCH_SIZE)
print("Batch candidates:", BATCH_SIZE_CANDIDATES)
print("Initial per-device batch / grad accum:", PER_DEVICE_BATCH, "/", GRAD_ACCUM)
print("Gradient checkpointing:", USE_GRADIENT_CHECKPOINTING)
print("Trainer checkpoints:", TRAINER_OUTPUT_DIR)
print("Final LoRA output:", FINAL_DIR)
print("Training metrics log:", MODEL_OUTPUT_DIR / "training_metrics.jsonl")


In [ ]:
def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def write_json(path, data):
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


def has_dataset_structure(root):
    root = Path(root)
    return (root / "train" / "metadata.jsonl").exists() and (root / "train" / "images").exists()


def resolve_dataset_root(path):
    path = Path(path).expanduser().resolve()
    if has_dataset_structure(path):
        return path
    for meta in sorted(path.rglob("train/metadata.jsonl")):
        root = meta.parent.parent
        if has_dataset_structure(root):
            return root
    raise FileNotFoundError(
        f"Could not find dataset structure under {path}. Expected train/images and train/metadata.jsonl."
    )


def configure_kaggle_credentials():
    if not KAGGLE_JSON_PATH.exists():
        raise FileNotFoundError(
            f"Missing Kaggle API file: {KAGGLE_JSON_PATH}. Put kaggle.json next to this notebook or set KAGGLE_JSON_PATH."
        )
    os.environ["KAGGLE_CONFIG_DIR"] = str(KAGGLE_JSON_PATH.parent)
    if os.name != "nt":
        try:
            os.chmod(KAGGLE_JSON_PATH, 0o600)
        except OSError as exc:
            print("Could not chmod kaggle.json; continuing:", repr(exc), flush=True)
    return KAGGLE_JSON_PATH


def download_dataset_if_needed():
    override = str(DATASET_ROOT_OVERRIDE or "").strip()
    if override:
        root = resolve_dataset_root(Path(override))
        print("Using existing dataset root:", root, flush=True)
        return root

    if FORCE_DATASET_DOWNLOAD and DATASET_DOWNLOAD_DIR.exists():
        print("Removing old dataset download dir:", DATASET_DOWNLOAD_DIR, flush=True)
        shutil.rmtree(DATASET_DOWNLOAD_DIR)
    DATASET_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

    try:
        root = resolve_dataset_root(DATASET_DOWNLOAD_DIR)
        print("Dataset already downloaded:", root, flush=True)
        return root
    except FileNotFoundError:
        pass

    configure_kaggle_credentials()
    print("Downloading Kaggle dataset:", KAGGLE_DATASET_SLUG, flush=True)
    from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()
    api.dataset_download_files(
        KAGGLE_DATASET_SLUG,
        path=str(DATASET_DOWNLOAD_DIR),
        unzip=True,
        quiet=False,
    )
    root = resolve_dataset_root(DATASET_DOWNLOAD_DIR)
    print("Dataset ready:", root, flush=True)
    return root


DATASET_ROOT = download_dataset_if_needed()


def find_fold_jsonl(fold):
    candidates = FOLD_FILE_CANDIDATES[fold]
    for name in candidates:
        path = FOLD_JSONL_ROOT / name
        if path.exists():
            return path
    available = ", ".join(p.name for p in sorted(FOLD_JSONL_ROOT.glob("*.jsonl")))
    raise FileNotFoundError(
        f"Missing fold {fold} jsonl in {FOLD_JSONL_ROOT}. Tried {candidates}. Available jsonl files: {available}"
    )


FOLD_JSONL_PATHS = {fold: find_fold_jsonl(fold) for fold in FOLD_FILE_CANDIDATES}
print("Fold manifests:")
for fold, path in FOLD_JSONL_PATHS.items():
    print(f"  FOLD{fold}: {path}")

IMAGE_SIZE_CACHE = {}
IMAGE_PATH_CACHE = {}
IMAGE_INDEX = None


def normalize_type(value):
    value = str(value or "handwritten").strip().lower()
    return value if value in VALID_TYPES else "handwritten"


def direct_image_candidates(file_name):
    rel = Path(str(file_name).replace("\\", "/"))
    base = rel.name
    rels = [
        rel,
        Path("images") / base,
        Path("train") / rel,
        Path("train") / "images" / base,
        Path("test") / rel,
        Path("test") / "images" / base,
        Path(base),
    ]
    for r in rels:
        yield DATASET_ROOT / r


def build_image_index():
    index = {}
    print("Building image basename index under train/images and test/images...", flush=True)
    for image_dir in [DATASET_ROOT / "train" / "images", DATASET_ROOT / "test" / "images"]:
        if not image_dir.exists():
            continue
        for path in image_dir.rglob("*"):
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                index.setdefault(path.name, path)
    print("Indexed image files:", len(index), flush=True)
    return index


def resolve_image_path(file_name):
    key = str(file_name).replace("\\", "/")
    if key in IMAGE_PATH_CACHE:
        return IMAGE_PATH_CACHE[key]
    for candidate in direct_image_candidates(key):
        if candidate.exists():
            IMAGE_PATH_CACHE[key] = str(candidate)
            return str(candidate)
    global IMAGE_INDEX
    if IMAGE_INDEX is None:
        IMAGE_INDEX = build_image_index()
    hit = IMAGE_INDEX.get(Path(key).name)
    if hit and hit.exists():
        IMAGE_PATH_CACHE[key] = str(hit)
        return str(hit)
    raise FileNotFoundError(f"Could not resolve image for {file_name} under dataset root {DATASET_ROOT}")


def image_size(path):
    path = str(path)
    if path not in IMAGE_SIZE_CACHE:
        with Image.open(path) as img:
            IMAGE_SIZE_CACHE[path] = img.size
    return IMAGE_SIZE_CACHE[path]


def clamp_box(box, width, height):
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        return None
    try:
        x1, y1, x2, y2 = [float(v) for v in box]
    except Exception:
        return None
    if x2 < x1:
        x1, x2 = x2, x1
    if y2 < y1:
        y1, y2 = y2, y1
    x1 = max(0, min(width - 1, int(round(x1))))
    y1 = max(0, min(height - 1, int(round(y1))))
    x2 = max(1, min(width, int(round(x2))))
    y2 = max(1, min(height, int(round(y2))))
    if x2 <= x1 or y2 <= y1:
        return None
    return [x1, y1, x2, y2]


def scaled_record_box(record, box, actual_w, actual_h):
    src_w = int(record.get("image_width") or actual_w)
    src_h = int(record.get("image_height") or actual_h)
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        return None
    vals = [float(v) for v in box]
    if src_w > 0 and src_h > 0 and (src_w != actual_w or src_h != actual_h):
        sx = actual_w / src_w
        sy = actual_h / src_h
        vals = [vals[0] * sx, vals[1] * sy, vals[2] * sx, vals[3] * sy]
    return clamp_box(vals, actual_w, actual_h)


def prompt_for_type(region_type):
    region_type = normalize_type(region_type)
    return CROP_PROMPTS.get(region_type, CROP_PROMPTS["handwritten"])


def make_ocr_samples_for_fold(fold):
    records = read_jsonl(FOLD_JSONL_PATHS[fold])
    samples = []
    skipped = Counter()
    for record_idx, record in enumerate(records):
        image_path = resolve_image_path(record.get("file_name"))
        actual_w, actual_h = image_size(image_path)
        for region_idx, region in enumerate(record.get("regions") or []):
            rtype = normalize_type(region.get("type"))
            if rtype not in OCR_TYPES:
                skipped[f"non_ocr_{rtype}"] += 1
                continue
            text = str(region.get("text") or "").strip()
            if not text:
                skipped["empty_text"] += 1
                continue
            box = scaled_record_box(record, region.get("bbox"), actual_w, actual_h)
            if box is None:
                skipped["bad_bbox"] += 1
                continue
            samples.append(
                {
                    "task": "crop_ocr",
                    "image_path": image_path,
                    "source_file_name": str(record.get("file_name") or ""),
                    "fold_part": int(fold),
                    "record_idx": int(record_idx),
                    "region_idx": int(region_idx),
                    "bbox": box,
                    "region_type": rtype,
                    "prompt": prompt_for_type(rtype),
                    "answer": text,
                    "source": str(record.get("source") or "unknown"),
                }
            )
    print(f"FOLD{fold}: pages={len(records)} samples={len(samples)} skipped={dict(skipped)}")
    return samples


def split_train_validation(rows, val_fraction, seed):
    val_fraction = float(val_fraction or 0.0)
    if val_fraction <= 0:
        return list(rows), []
    rng = random.Random(seed)
    grouped = defaultdict(list)
    for row in rows:
        grouped[(row.get("fold_part"), row.get("region_type"))].append(row)
    train_rows, val_rows = [], []
    for key, items in sorted(grouped.items(), key=lambda x: (x[0][0], x[0][1])):
        items = list(items)
        rng.shuffle(items)
        if len(items) <= 1:
            train_rows.extend(items)
            continue
        n_val = max(1, int(round(len(items) * val_fraction)))
        n_val = min(n_val, len(items) - 1)
        val_rows.extend(items[:n_val])
        train_rows.extend(items[n_val:])
    rng.shuffle(train_rows)
    rng.shuffle(val_rows)
    return train_rows, val_rows


fold_samples = {fold: make_ocr_samples_for_fold(fold) for fold in TRAIN_FOLDS}
all_samples = []
for fold in TRAIN_FOLDS:
    all_samples.extend(fold_samples[fold])
random.Random(SEED).shuffle(all_samples)
if TEST_MODE:
    all_samples = all_samples[:TEST_MAX_SAMPLES]
if not all_samples:
    raise RuntimeError("No OCR samples were built.")

train_samples, val_samples = split_train_validation(all_samples, VALIDATION_FRACTION, SEED)
train_ds = Dataset.from_list(train_samples)
eval_ds = Dataset.from_list(val_samples) if val_samples else None


def count_by(rows, key):
    return dict(sorted(Counter(str(row.get(key)) for row in rows).items()))


print("Dataset root:", DATASET_ROOT)
print("Samples total:", len(all_samples))
print("Train:", len(train_samples), count_by(train_samples, "fold_part"), count_by(train_samples, "region_type"))
print("Validation:", len(val_samples), count_by(val_samples, "fold_part"), count_by(val_samples, "region_type"))
print("Example sample:", {k: train_samples[0][k] for k in ["source_file_name", "bbox", "region_type", "answer"]})
write_json(
    MODEL_OUTPUT_DIR / "dataset_summary.json",
    {
        "model_id": MODEL_ID,
        "train_folds": TRAIN_FOLDS,
        "held_out_fold": HELD_OUT_FOLD,
        "dataset_root": str(DATASET_ROOT),
        "fold_jsonl_paths": {str(k): str(v) for k, v in FOLD_JSONL_PATHS.items()},
        "train_samples": len(train_samples),
        "validation_samples": len(val_samples),
        "train_by_fold": count_by(train_samples, "fold_part"),
        "train_by_region_type": count_by(train_samples, "region_type"),
    },
)


In [ ]:
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from qwen_vl_utils import process_vision_info
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig, TrainerCallback
from trl import SFTConfig, SFTTrainer


def model_dir_is_ready(path):
    path = Path(path).expanduser()
    return (path / "config.json").exists()


def download_qwen_model_if_needed():
    local_dir = Path(MODEL_DOWNLOAD_DIR).expanduser().resolve()
    if FORCE_MODEL_DOWNLOAD and local_dir.exists():
        print("Removing old Qwen model dir:", local_dir, flush=True)
        shutil.rmtree(local_dir)

    if model_dir_is_ready(local_dir):
        print("Qwen model already available:", local_dir, flush=True)
        return str(local_dir)

    for candidate in [NOTEBOOK_DIR / "Qwen3-VL-8B-Instruct", NOTEBOOK_DIR / "qwen3-vl-8b-instruct"]:
        candidate = Path(candidate).expanduser().resolve()
        if candidate != local_dir and model_dir_is_ready(candidate):
            print("Using existing local Qwen model:", candidate, flush=True)
            return str(candidate)

    print("Downloading Qwen model:", QWEN_MODEL_ID, "->", local_dir, flush=True)
    from huggingface_hub import snapshot_download

    local_dir.parent.mkdir(parents=True, exist_ok=True)
    try:
        snapshot_download(
            repo_id=QWEN_MODEL_ID,
            local_dir=str(local_dir),
            local_dir_use_symlinks=False,
            resume_download=True,
        )
    except TypeError:
        snapshot_download(repo_id=QWEN_MODEL_ID, local_dir=str(local_dir), resume_download=True)

    if not model_dir_is_ready(local_dir):
        raise FileNotFoundError(f"Downloaded Qwen model is missing config.json: {local_dir}")
    print("Qwen model ready:", local_dir, flush=True)
    return str(local_dir)


def find_model_id():
    return download_qwen_model_if_needed()


def resolve_start_lora_dir(path):
    path = str(path or "").strip()
    if not path:
        return None
    p = Path(path).expanduser()
    if not (p / "adapter_config.json").exists():
        raise FileNotFoundError(f"No adapter_config.json found in START_LORA_DIR={p}")
    if not (p / "adapter_model.safetensors").exists() and not (p / "adapter_model.bin").exists():
        raise FileNotFoundError(f"No adapter_model.safetensors or adapter_model.bin found in START_LORA_DIR={p}")
    return p


def configure_processor(processor):
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
    processor.tokenizer.padding_side = "left"
    return processor


def force_model_dtype_config(model, torch_dtype):
    model.config.torch_dtype = torch_dtype
    dtype_name = "bfloat16" if torch_dtype is torch.bfloat16 else "float16"
    for attr in ("text_config", "vision_config"):
        cfg = getattr(model.config, attr, None)
        if cfg is not None:
            cfg.torch_dtype = torch_dtype
            if hasattr(cfg, "dtype"):
                cfg.dtype = dtype_name


def make_lora_config():
    kwargs = dict(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
    )
    try:
        return LoraConfig(**kwargs, use_rslora=True)
    except TypeError:
        return LoraConfig(**kwargs)


def build_max_memory(reserve_gib=5):
    max_memory = {}
    for idx in range(torch.cuda.device_count()):
        total_gib = torch.cuda.get_device_properties(idx).total_memory // 1024**3
        max_memory[idx] = f"{max(8, total_gib - reserve_gib)}GiB"
    return max_memory or None


model_id = find_model_id()
start_lora_dir = resolve_start_lora_dir(START_LORA_DIR)
print("Base model:", model_id)
print("Start LoRA:", start_lora_dir)

processor = configure_processor(AutoProcessor.from_pretrained(model_id, trust_remote_code=True))

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=TORCH_DTYPE,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

base_model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map=MODEL_DEVICE_MAP,
    max_memory=build_max_memory(),
    quantization_config=quantization_config,
    dtype=TORCH_DTYPE,
    trust_remote_code=True,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
force_model_dtype_config(base_model, TORCH_DTYPE)
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=USE_GRADIENT_CHECKPOINTING)
if USE_GRADIENT_CHECKPOINTING:
    try:
        base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    except TypeError:
        base_model.gradient_checkpointing_enable()
else:
    print("Gradient checkpointing disabled for speed.", flush=True)

if start_lora_dir is not None:
    model = PeftModel.from_pretrained(base_model, str(start_lora_dir), is_trainable=True)
else:
    print("Creating a fresh LoRA adapter from scratch.", flush=True)
    model = get_peft_model(base_model, make_lora_config())

for _, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
model.print_trainable_parameters()


In [ ]:
def crop_with_padding(image_path, bbox, pad_ratio=CROP_PAD_RATIO):
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        w, h = img.size
        x1, y1, x2, y2 = [int(v) for v in bbox]
        pad = int(round(max(x2 - x1, y2 - y1) * float(pad_ratio)))
        x1 = max(0, x1 - pad)
        y1 = max(0, y1 - pad)
        x2 = min(w, x2 + pad)
        y2 = min(h, y2 + pad)
        if x2 <= x1 or y2 <= y1:
            return img.copy()
        return img.crop((x1, y1, x2, y2)).copy()


def build_messages(sample):
    crop = crop_with_padding(sample["image_path"], sample["bbox"])
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": crop, "max_pixels": MAX_PIXELS_CROP},
                {"type": "text", "text": sample["prompt"]},
            ],
        },
        {"role": "assistant", "content": [{"type": "text", "text": sample["answer"]}]},
    ]


def encode_marker(tokenizer):
    try:
        return tokenizer.encode("<|im_start|>assistant\n", allowed_special="all", add_special_tokens=False)
    except TypeError:
        return tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)


ASSISTANT_MARKER = encode_marker(processor.tokenizer)


def data_collator(examples):
    messages_list = [build_messages(ex) for ex in examples]
    texts = [processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=False) for msg in messages_list]
    image_inputs, video_inputs = process_vision_info(messages_list)
    batch = processor(
        text=texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        return_tensors="pt",
    )

    labels = batch["input_ids"].clone()
    pad_id = processor.tokenizer.pad_token_id
    if pad_id is not None:
        labels[labels == pad_id] = -100

    eos_id = processor.tokenizer.eos_token_id
    for i in range(labels.shape[0]):
        ids = batch["input_ids"][i].tolist()
        start = -1
        for j in range(0, len(ids) - len(ASSISTANT_MARKER) + 1):
            if ids[j:j + len(ASSISTANT_MARKER)] == ASSISTANT_MARKER:
                start = j + len(ASSISTANT_MARKER)
                break
        actual_len = int(batch["attention_mask"][i].sum().item())
        truncated = actual_len >= MAX_SEQ_LENGTH and (eos_id is None or ids[actual_len - 1] != eos_id)
        if start >= 0 and not truncated:
            labels[i, :start] = -100
        else:
            labels[i, :] = -100

    batch["labels"] = labels
    for key, value in list(batch.items()):
        if isinstance(value, torch.Tensor) and value.dtype == torch.float32:
            batch[key] = value.to(TORCH_DTYPE)
    return batch


In [ ]:
import inspect

from IPython.display import clear_output
from tqdm.auto import tqdm
from transformers import PrinterCallback, ProgressCallback


class NotebookProgressCallback(TrainerCallback):
    def __init__(self, name):
        self.name = name
        self.pbar = None
        self.last_step = 0

    def on_train_begin(self, args, state, control, **kwargs):
        clear_output(wait=True)
        self.last_step = int(state.global_step or 0)
        self.pbar = tqdm(
            total=int(state.max_steps or 0),
            initial=self.last_step,
            desc=self.name,
            dynamic_ncols=True,
            leave=True,
        )

    def on_log(self, args, state, control, logs=None, **kwargs):
        if self.pbar is None:
            return
        step = int(state.global_step or 0)
        if step > self.last_step:
            self.pbar.update(step - self.last_step)
            self.last_step = step
        logs = logs or {}
        keep = {k: logs[k] for k in ["loss", "eval_loss", "learning_rate", "grad_norm"] if k in logs}
        if keep:
            self.pbar.set_postfix(keep, refresh=True)

    def on_step_end(self, args, state, control, **kwargs):
        if self.pbar is None:
            return
        step = int(state.global_step or 0)
        if step > self.last_step:
            self.pbar.update(step - self.last_step)
            self.last_step = step

    def on_epoch_end(self, args, state, control, **kwargs):
        control.should_save = True
        return control

    def on_save(self, args, state, control, **kwargs):
        if self.pbar is not None:
            self.pbar.write(f"saved checkpoint-{state.global_step}")

    def on_train_end(self, args, state, control, **kwargs):
        if self.pbar is not None:
            step = int(state.global_step or 0)
            if step > self.last_step:
                self.pbar.update(step - self.last_step)
            self.pbar.close()
            self.pbar = None


class OOMRecoverySFTTrainer(SFTTrainer):
    def training_step(self, model, inputs, num_items_in_batch=None):
        try:
            try:
                loss = super().training_step(model, inputs, num_items_in_batch=num_items_in_batch)
            except TypeError:
                loss = super().training_step(model, inputs)
            if self.args.device != loss.device:
                loss = loss.to(self.args.device)
            return loss
        except torch.cuda.OutOfMemoryError:
            print("OOM: skipping one batch after clearing cache.", flush=True)
            for p in model.parameters():
                p.grad = None
            torch.cuda.empty_cache()
            gc.collect()
            return torch.tensor(0.0, device=self.args.device)


def find_latest_checkpoint(root):
    root = Path(root)
    if not root.exists():
        return None
    checkpoints = []
    for path in root.glob("checkpoint-*"):
        try:
            step = int(path.name.rsplit("-", 1)[-1])
        except ValueError:
            continue
        if (path / "trainer_state.json").exists():
            checkpoints.append((step, path))
    if not checkpoints:
        return None
    return str(max(checkpoints, key=lambda item: item[0])[1])


def resolve_resume_checkpoint(output_dir):
    if not RESUME_TRAINING:
        return None
    if RESUME_CHECKPOINT_DIR:
        resume_root = Path(RESUME_CHECKPOINT_DIR).expanduser()
        if (resume_root / "trainer_state.json").exists():
            return str(resume_root)
        latest = find_latest_checkpoint(resume_root)
        if latest:
            return latest
        raise FileNotFoundError(f"No checkpoint-* with trainer_state.json found under {resume_root}")
    return find_latest_checkpoint(output_dir)


def sft_config_supports_arg(name):
    try:
        return name in inspect.signature(SFTConfig.__init__).parameters
    except Exception:
        return False


def filter_sft_config_kwargs(kwargs):
    try:
        params = inspect.signature(SFTConfig.__init__).parameters
    except Exception:
        return dict(kwargs)
    if any(param.kind == inspect.Parameter.VAR_KEYWORD for param in params.values()):
        return dict(kwargs)
    filtered = {k: v for k, v in kwargs.items() if k in params}
    skipped = sorted(set(kwargs) - set(filtered))
    if skipped:
        print("SFTConfig skipped unsupported args:", skipped, flush=True)
    return filtered


class MetricsFileCallback(TrainerCallback):
    def __init__(self, path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def on_train_begin(self, args, state, control, **kwargs):
        if int(state.global_step or 0) == 0:
            self.path.write_text("", encoding="utf-8")
        else:
            self.path.touch(exist_ok=True)

    def on_log(self, args, state, control, logs=None, **kwargs):
        row = {
            "time": time.strftime("%Y-%m-%d %H:%M:%S"),
            "step": int(state.global_step or 0),
            "epoch": float(state.epoch or 0.0),
        }
        row.update(logs or {})
        with self.path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def get_input_device(model):
    for param in model.parameters():
        if param.device.type == "cuda":
            return param.device
    return torch.device("cuda:0")


def clear_cuda_memory():
    for param in model.parameters():
        param.grad = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def select_probe_examples(rows, batch_size):
    ranked = sorted(
        rows,
        key=lambda row: len(str(row.get("prompt", ""))) + len(str(row.get("answer", ""))),
        reverse=True,
    )
    if len(ranked) >= batch_size:
        return ranked[:batch_size]
    repeats = math.ceil(batch_size / max(1, len(ranked)))
    return (ranked * repeats)[:batch_size]


def batch_to_device(batch, device):
    moved = {}
    for key, value in batch.items():
        moved[key] = value.to(device) if isinstance(value, torch.Tensor) else value
    return moved


def probe_batch_size(batch_size):
    clear_cuda_memory()
    probe_examples = select_probe_examples(train_samples, batch_size)
    input_device = get_input_device(model)
    model.train()
    try:
        batch = batch_to_device(data_collator(probe_examples), input_device)
        with torch.amp.autocast("cuda", dtype=TORCH_DTYPE):
            outputs = model(**batch)
            loss = outputs.loss
        loss.backward()
        torch.cuda.synchronize()
        peak_gib = torch.cuda.max_memory_allocated() / 1024**3
        print(f"Batch probe OK: batch={batch_size}, peak_vram={peak_gib:.1f}GB", flush=True)
        return True, peak_gib
    except torch.cuda.OutOfMemoryError:
        print(f"Batch probe OOM: batch={batch_size}", flush=True)
        return False, None
    except RuntimeError as exc:
        if "out of memory" in str(exc).lower():
            print(f"Batch probe OOM: batch={batch_size}", flush=True)
            return False, None
        raise
    finally:
        clear_cuda_memory()


def tune_batch_size_for_speed():
    global PER_DEVICE_BATCH, GRAD_ACCUM
    if not AUTO_TUNE_BATCH_SIZE:
        if MAX_SPEED_MODE:
            GRAD_ACCUM = max(1, int(GRAD_ACCUM))
        else:
            GRAD_ACCUM = max(1, int(GRAD_ACCUM))
        print(
            f"Batch autotune disabled: batch={PER_DEVICE_BATCH}, grad_accum={GRAD_ACCUM}, "
            f"effective_batch={PER_DEVICE_BATCH * GRAD_ACCUM}",
            flush=True,
        )
        return

    candidates = sorted({int(x) for x in BATCH_SIZE_CANDIDATES if int(x) > 0}, reverse=True)
    if not candidates:
        raise ValueError("BATCH_SIZE_CANDIDATES must contain at least one positive integer")
    print("Autotuning per-device batch candidates:", candidates, flush=True)
    for candidate in candidates:
        ok, _ = probe_batch_size(candidate)
        if ok:
            PER_DEVICE_BATCH = candidate
            GRAD_ACCUM = 1 if MAX_SPEED_MODE else max(1, math.ceil(TARGET_EFFECTIVE_BATCH / PER_DEVICE_BATCH))
            print(
                f"Selected batch={PER_DEVICE_BATCH}, grad_accum={GRAD_ACCUM}, "
                f"effective_batch={PER_DEVICE_BATCH * GRAD_ACCUM}, max_speed={MAX_SPEED_MODE}",
                flush=True,
            )
            return
    raise RuntimeError("No batch size candidate fit in GPU memory, including batch=1.")


tune_batch_size_for_speed()


config_kwargs = dict(
    output_dir=str(TRAINER_OUTPUT_DIR),
    overwrite_output_dir=True,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    logging_strategy="steps",
    logging_first_step=True,
    logging_steps=LOGGING_STEPS,
    disable_tqdm=True,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
    dataloader_num_workers=DATALOADER_NUM_WORKERS,
    dataloader_pin_memory=False,
    dataloader_persistent_workers=DATALOADER_NUM_WORKERS > 0,
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
)
if eval_ds is not None:
    if sft_config_supports_arg("eval_strategy"):
        config_kwargs["eval_strategy"] = "epoch"
    elif sft_config_supports_arg("evaluation_strategy"):
        config_kwargs["evaluation_strategy"] = "epoch"
    if sft_config_supports_arg("do_eval"):
        config_kwargs["do_eval"] = True
    print("Validation enabled: eval loss once per epoch.", flush=True)
else:
    print("Validation disabled.", flush=True)

training_args = SFTConfig(**filter_sft_config_kwargs(config_kwargs))
trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=data_collator,
    callbacks=[NotebookProgressCallback(RUN_NAME), MetricsFileCallback(MODEL_OUTPUT_DIR / "training_metrics.jsonl")],
)
if eval_ds is not None:
    trainer_kwargs["eval_dataset"] = eval_ds

trainer = OOMRecoverySFTTrainer(**trainer_kwargs)
trainer.remove_callback(PrinterCallback)
trainer.remove_callback(ProgressCallback)

resume_checkpoint = resolve_resume_checkpoint(TRAINER_OUTPUT_DIR)
print("Resume checkpoint:", resume_checkpoint)
trainer.train(resume_from_checkpoint=resume_checkpoint)

trainer.model.save_pretrained(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)
prompt_config = {
    "prompt_version": PROMPT_VERSION,
    "crop_prompts": CROP_PROMPTS,
    "max_pixels_crop": MAX_PIXELS_CROP,
}
write_json(FINAL_DIR / "rukopys_prompt_config.json", prompt_config)

train_config = {
    "stage": "oof_v2_qwen3vl_8b_ocr_crop_on_the_fly",
    "model_id": MODEL_ID,
    "train_folds": TRAIN_FOLDS,
    "held_out_fold": HELD_OUT_FOLD,
    "run_name": RUN_NAME,
    "prompt_version": PROMPT_VERSION,
    "base_model": str(model_id),
    "qwen_model_id": QWEN_MODEL_ID,
    "model_download_dir": str(MODEL_DOWNLOAD_DIR),
    "start_lora": str(start_lora_dir) if start_lora_dir else None,
    "torch_dtype": "bfloat16" if USE_BF16 else "float16",
    "output_root": str(OUTPUT_ROOT),
    "trainer_output_dir": str(TRAINER_OUTPUT_DIR),
    "final_dir": str(FINAL_DIR),
    "train_samples": len(train_samples),
    "validation_samples": len(val_samples),
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "max_speed_mode": MAX_SPEED_MODE,
    "auto_tune_batch_size": AUTO_TUNE_BATCH_SIZE,
    "batch_size_candidates": BATCH_SIZE_CANDIDATES,
    "target_effective_batch": TARGET_EFFECTIVE_BATCH,
    "per_device_batch": PER_DEVICE_BATCH,
    "gradient_accumulation_steps": GRAD_ACCUM,
    "effective_batch": PER_DEVICE_BATCH * GRAD_ACCUM,
    "training_metrics_log": str(MODEL_OUTPUT_DIR / "training_metrics.jsonl"),
    "gradient_checkpointing": USE_GRADIENT_CHECKPOINTING,
    "learning_rate": LEARNING_RATE,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "max_pixels_crop": MAX_PIXELS_CROP,
    "max_seq_length": MAX_SEQ_LENGTH,
    "resume_from_checkpoint": resume_checkpoint,
}
write_json(FINAL_DIR / "rukopys_oof_v2_ocr_train_config.json", train_config)
print("Saved final OCR LoRA adapter to:", FINAL_DIR)
print("Trainer checkpoints are in:", TRAINER_OUTPUT_DIR)
print("Training metrics log:", MODEL_OUTPUT_DIR / "training_metrics.jsonl")


In [ ]:
# Optional quick sanity check on one validation crop. This is not a leaderboard estimate.
RUN_QUICK_SANITY = True

if RUN_QUICK_SANITY and (val_samples or train_samples):
    model.eval()
    sample = (val_samples or train_samples)[0]
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": crop_with_padding(sample["image_path"], sample["bbox"]), "max_pixels": MAX_PIXELS_CROP},
                {"type": "text", "text": sample["prompt"]},
            ],
        }
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    input_device = next((p.device for p in model.parameters() if p.device.type == "cuda"), torch.device("cuda:0"))
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(input_device)
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=TORCH_DTYPE):
        out = model.generate(**inputs, max_new_tokens=192, do_sample=False, num_beams=1)
    trimmed = out[:, inputs.input_ids.shape[1]:]
    pred = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    print("Image:", sample["source_file_name"])
    print("Region:", sample["bbox"], sample["region_type"])
    print("Target:", sample["answer"])
    print("Pred:", pred)
